# Week 4 Lab：Flow Matching 與 ODE 取樣

## 學習目標
- 從 endpoint pair 建立可控曲率的 conditional path 與速度 target。
- 由解析旋轉／縮放 coupling 得到 marginal velocity field。
- 用 log-log 圖比較 Euler 與 Heun 的全域誤差階數。

> **誠實註記**：本 notebook 使用可解析的旋轉／縮放 flow 當作完美 surrogate，不載入也不假裝有神經網路 checkpoint。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 2718
rng = np.random.default_rng(SEED)
b = np.array([0.7, -0.25])
THETA = .9

def rotate(x, angle):
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]])
    return np.asarray(x) @ R.T

def conditional_path(z, t):
    scale = 1 + .25 * np.sin(np.pi * t)
    return scale * rotate(z, THETA * t ** 2) + t * b

def endpoint(z):
    return conditional_path(z, 1.0)

z = rng.standard_normal((180, 2))
x1 = endpoint(z)
fig, ax = plt.subplots(figsize=(6, 5))
for i in range(30):
    path = np.array([conditional_path(z[i], t) for t in np.linspace(0, 1, 30)])
    ax.plot(path[:, 0], path[:, 1], color='0.72', lw=1)
ax.scatter(z[:, 0], z[:, 1], s=9, alpha=.45, label='source $p_0$')
ax.scatter(x1[:, 0], x1[:, 1], s=9, alpha=.45, label='target $p_1$')
ax.set(aspect='equal', title='Conditional FM paths and endpoint pairs')
ax.legend()
plt.show()

In [ ]:
def velocity(x, t):
    # Invert the analytic conditional path, then evaluate its time derivative.
    x = np.atleast_2d(x)
    scale = 1 + .25 * np.sin(np.pi * t)
    scale_dot = .25 * np.pi * np.cos(np.pi * t)
    angle, angle_dot = THETA * t ** 2, 2 * THETA * t
    x0 = rotate((x - t * b) / scale, -angle)
    rotated = rotate(x0, angle)
    J_rotated = np.stack([-rotated[:, 1], rotated[:, 0]], axis=-1)
    return scale_dot * rotated + scale * angle_dot * J_rotated + b

grid = np.linspace(-3, 3, 21)
gx, gy = np.meshgrid(grid, grid)
p = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, t in zip(axes, [0.0, 0.5, 1.0]):
    v = velocity(p, t)
    ax.quiver(p[:, 0], p[:, 1], v[:, 0], v[:, 1], angles='xy', scale=32, width=.004)
    ax.set(title=f'marginal velocity, t={t:.1f}', xlim=(-3, 3), ylim=(-3, 3), aspect='equal')
plt.show()

In [ ]:
def integrate(x0, n_steps, method='euler'):
    x = np.array(x0, dtype=float, copy=True)
    dt = 1.0 / n_steps
    for k in range(n_steps):
        t = k * dt
        v0 = velocity(x, t)
        if method == 'euler':
            x = x + dt * v0
        elif method == 'heun':
            predictor = x + dt * v0
            x = x + 0.5 * dt * (v0 + velocity(predictor, t + dt))
        else:
            raise ValueError('method must be euler or heun')
    return x

test = rng.standard_normal((400, 2))
truth = endpoint(test)
steps = np.array([2, 4, 8, 16, 32, 64])
errors = {}
for method in ['euler', 'heun']:
    errors[method] = np.array([np.sqrt(np.mean((integrate(test, n, method) - truth) ** 2)) for n in steps])
    slope = np.polyfit(np.log(steps), np.log(errors[method]), 1)[0]
    print(f'{method:5s} log-log slope = {slope:.2f}')

fig, ax = plt.subplots(figsize=(6, 4))
for method in errors:
    ax.loglog(steps, errors[method], 'o-', label=method)
ax.set(xlabel='ODE steps', ylabel='endpoint RMSE', title='Sampler order changes the slope')
ax.grid(True, which='both', alpha=.25)
ax.legend()
plt.show()

In [ ]:
start = np.array([[-1.7, -1.1], [0.2, 1.7], [1.5, -0.7]])
fig, ax = plt.subplots(figsize=(6, 5))
for method, style in [('euler', '--'), ('heun', '-')]:
    for n in [4, 12]:
        path = [start.copy()]
        x = start.copy()
        dt = 1 / n
        for k in range(n):
            t = k * dt
            v0 = velocity(x, t)
            x = x + dt * v0 if method == 'euler' else x + .5 * dt * (v0 + velocity(x + dt * v0, t + dt))
            path.append(x.copy())
        path = np.stack(path)
        for j in range(len(start)):
            ax.plot(path[:, j, 0], path[:, j, 1], style, alpha=.7, label=f'{method}, {n} steps' if j == 0 else None)
ax.scatter(endpoint(start)[:, 0], endpoint(start)[:, 1], marker='x', s=80, c='black', label='exact endpoint')
ax.set(aspect='equal', title='Numerical trajectories')
ax.legend(ncol=2, fontsize=8)
plt.show()

## 讀者練習 / TODO
把 source sampling 換成均勻圓環，但保留同一個 `endpoint` coupling。哪些函數完全不用改？再把 `A` 改得更接近奇異矩陣，觀察誤差曲線的**高度**與**斜率**是否一起改變。